In [3]:

"""
PIPELINE STAGE: Multimodal Data Integration & Temporal Broadcasting
PERIOD: January 2020 - December 2024
LIBRARIES: os, pandas

1. OBJECTIVE:
   Execute the final cross-format data synthesis, integrating annual demographic denominators 
   (parsed from CSV) into the monthly energy database (mined from unstructured Word/Excel). 
   This stage creates a unified relational schema primed for high-resolution per-capita modeling.

2. FILE ARCHITECTURE:
   - Input 1 (Energy): os.path.join("..", "..", "processed_data", "steps", "07_final_energy_consolidation.xlsx")
   - Input 2 (Population): os.path.join("..", "..", "processed_data", "steps", "08_a_cleaned_population.xlsx")
   - Output (Merged): os.path.join("..", "..", "processed_data", "steps", "09_energy_and_population.xlsx")
   - System: Dynamically generates the output directory structure if it does not exist.

3. TECHNICAL LOGIC & TEMPORAL BROADCASTING:
   - Granularity Alignment: Annual demographic data is broadcasted across the 12-month 
     energy granularity. A single yearly population value dynamically replicates for all 
     months within that specific year.
   - Composite Key Integration: Executes a precision 'Left Join' using [Plate] and [Year] 
     as primary keys to explicitly bypass orthographic inconsistencies in province names.
   - Selective Appending: Merges ONLY the 'Population' column from the demographic dataset 
     to prevent schema bloat and duplicate columns (e.g., Province_x, Province_y).

4. SCHEMA ARCHITECTURE & INTEGRITY:
   - Header Sanitization: Automatically strips leading/trailing whitespaces from all 
     column headers in both datasets prior to merge operations.
   - Type Enforcement: Strictly casts join-keys ('Plate', 'Year') to Integers in both DataFrames 
     to neutralize float/string mismatches and prevent null propagation.
   - Validation Check: Implements an automated audit post-merge to count missing values (isna), 
     warning the user immediately if any rows failed to map a 'Population' value.
"""


import pandas as pd
import os

def enerji_ve_nufus_birlestir():
    # 1. Dosya Yolları
    # Ana enerji dosyası (processed_data/steps içindeki son halini referans alıyoruz)
    ENERGY_PATH = os.path.join("..", "..", "processed_data", "steps","07_final_energy_consolidation.xlsx")
    # Yeni temizlediğimiz nüfus dosyası
    POP_PATH = os.path.join("..", "..", "processed_data", "steps", "08_a_cleaned_population.xlsx")
    # Final çıktı yolu
    FINAL_OUT = os.path.join("..", "..", "processed_data", "steps", "09_energy_and_population.xlsx")
    
    try:
        # 2. Dosyaları Oku
        print("Tablolar okunuyor...")
        df_energy = pd.read_excel(ENERGY_PATH)
        df_pop = pd.read_excel(POP_PATH)

        # Sütun isimlerindeki olası boşlukları temizleyelim
        df_energy.columns = [c.strip() for c in df_energy.columns]
        df_pop.columns = [c.strip() for c in df_pop.columns]

        # 3. Veri Tiplerini Eşitle (Kritik: Plate ve Year her iki tabloda da integer olmalı)
        df_energy['Plate'] = df_energy['Plate'].astype(int)
        df_energy['Year'] = df_energy['Year'].astype(int)
        df_pop['Plate'] = df_pop['Plate'].astype(int)
        df_pop['Year'] = df_pop['Year'].astype(int)

        # 4. Birleştirme (Left Join)
        # Nüfus tablosundan sadece eşleşme anahtarları ve Population sütununu alıyoruz
        print("Plate ve Year bazında nüfus verileri entegre ediliyor...")
        final_df = pd.merge(
            df_energy, 
            df_pop[['Plate', 'Year', 'Population']], 
            on=['Plate', 'Year'], 
            how='left'
        )

        # 5. Son Kontroller
        # Eğer bazı satırlarda nüfus boş kaldıysa uyarı ver
        missing_count = final_df['Population'].isna().sum()
        if missing_count > 0:
            print(f"UYARI: {missing_count} satırda nüfus verisi eşleşmedi! Lütfen eksik yılları/plakaları kontrol edin.")

        # 6. Kaydet
        os.makedirs(os.path.dirname(FINAL_OUT), exist_ok=True)
        final_df.to_excel(FINAL_OUT, index=False)
        
        print("\n" + "="*40)
        print(f"İŞLEM BAŞARIYLA TAMAMLANDI")
        print(f"Yeni Sütun: Population (En sona eklendi)")
        print(f"Toplam Satır: {len(final_df)}")
        print(f"Dosya: {FINAL_OUT}")
        print("="*40)

    except Exception as e:
        print(f"HATA OLUŞTU: {e}")

if __name__ == "__main__":
    enerji_ve_nufus_birlestir()

Tablolar okunuyor...
Plate ve Year bazında nüfus verileri entegre ediliyor...

İŞLEM BAŞARIYLA TAMAMLANDI
Yeni Sütun: Population (En sona eklendi)
Toplam Satır: 4860
Dosya: ..\..\processed_data\steps\09_energy_and_population.xlsx
